# Accessing data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Putting confidence interval shading on the anomaly plots
- Using the existing code from 'anomaly_stratify_hour.ipynb'
- Adding in confidence interval shading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def compare_holiday_only_window(demand, info, station, years, holiday_func, holiday_name,
                                save_dir=None):
    """
    Plot demand anomalies for a holiday (24-hour profile),
    using the ±30-day window around the holiday as baseline,
    stratified by hour of day.
    Includes robust 95% confidence interval shading.
    """

    # --- Prepare hourly demand data ---
    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    # --- Collect ±30-day windows around the holiday ---
    windows = []
    for year in years:
        ref_date = holiday_func(year)
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)
        window = hourly.loc[start:end].copy()
        windows.append(window)

    combined = pd.concat(windows)

    # --- Compute baseline stratified by hour ---
    combined["hour"] = combined.index.hour
    baseline_by_hour = combined.groupby("hour")[station].mean()

    # --- Compute anomalies relative to hour-specific baseline ---
    anomalies = combined.copy()
    anomalies["anomaly"] = anomalies[station] - anomalies["hour"].map(baseline_by_hour)

    # --- Extract holiday anomaly profile (24 hours) for each year ---
    holiday_hours = []
    for year in years:
        ref_date = holiday_func(year)
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")

        # Extract anomalies for the 24-hour window
        daily = anomalies["anomaly"].reindex(expected_hours)

        # Reset index to match length, then force 24 rows
        daily.index = range(len(daily))
        daily = daily.reindex(range(24))

        holiday_hours.append(daily)

    # --- Build matrix of per-year holiday anomaly profiles ---
    holiday_matrix = pd.concat(holiday_hours, axis=1)

    # --- Mean profile ---
    holiday_profile = holiday_matrix.mean(axis=1)

    # --- 95% CI (NaN-safe, only where >=2 valid years) ---
    std_profile = holiday_matrix.std(axis=1)
    n_years = holiday_matrix.count(axis=1)  # count non-NaN per hour
    ci95 = 1.96 * std_profile / np.sqrt(n_years)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12,5))

    # Confidence interval shading (only where CI is valid)
    ax.fill_between(
        holiday_profile.index,
        holiday_profile - ci95,
        holiday_profile + ci95,
        where=~ci95.isna(),
        color="red",
        alpha=0.2,
        label="95% CI"
    )

    # Mean anomaly line
    ax.plot(
        holiday_profile.index,
        holiday_profile.values,
        color="red",
        linewidth=2,
        marker="o",
        label=f"{holiday_name} (±30-Day Window)"
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.grid(axis='y', linestyle='-', linewidth=0.5, color='gray', alpha=0.3)
    ax.set_xticks(np.arange(0, 24, 1))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} Demand Anomaly: {holiday_name} ({years[0]}–{years[-1]}, ±30-Day Window)",
        fontsize=14
    )
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Electricity Demand Anomaly")
    ax.legend()
    fig.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        outpath = os.path.join(save_dir, f"{station}_{holiday_name}.pdf")
        fig.savefig(outpath, dpi=150, format="pdf")
    else:
        plt.close()

## Testing function with just BLAKE

In [ ]:
compare_holiday_only_window(
    demand=demand,
    info=info,
    station="BLAKE",
    years=[2012, 2013],
    holiday_func=lambda y: pd.Timestamp(y, 12, 25),
    holiday_name="Christmas Day"
)

## Looping through all holidays for all substations
- Saves each substation in a separate folder
- Creates a pdf of each holiday containing every 2 year interval with 95% confidence interval

In [ ]:
import os

BASE_DIR = "/home/565/pv3484/aus_substation_electricity/figures/anomaly_demand/confidence_interval"

years = list(range(2004, 2018))

for station in demand.columns:
    
    station_dir = os.path.join(BASE_DIR, station)
    os.makedirs(station_dir, exist_ok=True)

    for holiday_name, holiday_func in HOLIDAYS.items():

        compare_holiday_only_window(
            demand=demand,
            info=info,
            station=station,
            years=years,
            holiday_func=holiday_func,
            holiday_name=holiday_name,
            save_dir=station_dir
        )